# Phase 1 Repair: XGBoost on Colab GPU

This notebook mounts Google Drive, loads `supernova_dataset.npz`, runs the repaired XGBoost pipeline with a train/validation split, and saves outputs into phase-1 repair folders so the old paper outputs are not overwritten.

In [ ]:
%pip install -q xgboost optuna imbalanced-learn scikit-learn joblib pandas numpy

In [ ]:
import os
import json
import numpy as np
import optuna
import xgboost as xgb
from joblib import dump
from google.colab import drive
from imblearn.over_sampling import SMOTE
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
drive.mount('/content/drive')

In [ ]:
DATA_PATH = '/content/drive/MyDrive/data/supernova_dataset.npz'
MODEL_ROOT = '/content/drive/MyDrive/models/phase1_repair/xgboost'
RESULTS_DIR = '/content/drive/MyDrive/results/phase1_repair'
N_TRIALS = 50
RANDOM_STATE = 42

assert os.path.exists(DATA_PATH), f'Dataset not found: {DATA_PATH}'
os.makedirs(MODEL_ROOT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('DATA_PATH =', DATA_PATH)
print('MODEL_ROOT =', MODEL_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)

In [ ]:
data = np.load(DATA_PATH, allow_pickle=True)
print('Keys:', sorted(data.files))

X_train = data['X_train']
Y_train = data['Y_train']
X_test = data['X_test']
Y_test = data['Y_test']

print('X_train shape:', X_train.shape)
print('Y_train shape:', Y_train.shape)
print('X_test shape:', X_test.shape)
print('Y_test shape:', Y_test.shape)

In [ ]:
try:
    import subprocess
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
except Exception as exc:
    print('Could not query GPU:', exc)

In [ ]:
def train_xgboost_model(X_train, Y_train, X_test, Y_test, use_smote=False, n_trials=50):
    subfolder = 'with_SMOTE' if use_smote else 'without_SMOTE'
    model_dir = os.path.join(MODEL_ROOT, subfolder)
    os.makedirs(model_dir, exist_ok=True)

    Y_train_labels = np.argmax(Y_train, axis=1)
    Y_test_labels = np.argmax(Y_test, axis=1)

    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_flat,
        Y_train_labels,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=Y_train_labels,
    )

    if use_smote:
        smote = SMOTE(random_state=RANDOM_STATE)
        X_tr, y_tr = smote.fit_resample(X_tr, y_tr)

    tuning_scaler = StandardScaler()
    X_tr_scaled = tuning_scaler.fit_transform(X_tr)
    X_val_scaled = tuning_scaler.transform(X_val)

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'gamma': trial.suggest_float('gamma', 0.0, 5.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
            'random_state': RANDOM_STATE,
            'n_jobs': -1,
            'tree_method': 'hist',
            'device': 'cuda',
            'eval_metric': 'logloss'
        }

        model = xgb.XGBClassifier(**params)
        model.fit(X_tr_scaled, y_tr)
        preds = model.predict(X_val_scaled)
        return f1_score(y_val, preds, average='weighted')

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)

    best_params = study.best_params
    final_params = {
        **best_params,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': 'logloss'
    }

    X_trainval_raw = X_train_flat
    y_trainval_raw = Y_train_labels
    if use_smote:
        smote = SMOTE(random_state=RANDOM_STATE)
        X_trainval_raw, y_trainval_raw = smote.fit_resample(X_trainval_raw, y_trainval_raw)

    final_scaler = StandardScaler()
    X_trainval_scaled = final_scaler.fit_transform(X_trainval_raw)
    X_test_scaled = final_scaler.transform(X_test_flat)

    best_model = xgb.XGBClassifier(**final_params)
    best_model.fit(X_trainval_scaled, y_trainval_raw)

    dump(final_scaler, os.path.join(model_dir, 'scaler.pkl'))
    dump(best_model, os.path.join(model_dir, 'model.pkl'))

    preds = best_model.predict(X_test_scaled)
    probs = best_model.predict_proba(X_test_scaled)[:, 1]

    results = {
        'model': f'xgboost_{subfolder}',
        'precision': float(precision_score(Y_test_labels, preds, average='weighted')),
        'recall': float(recall_score(Y_test_labels, preds, average='weighted')),
        'f1_score': float(f1_score(Y_test_labels, preds, average='weighted')),
        'roc_auc': float(roc_auc_score(Y_test_labels, probs)),
        'pr_auc': float(average_precision_score(Y_test_labels, probs)),
        'best_params': best_params
    }

    with open(os.path.join(model_dir, 'results.json'), 'w') as f:
        json.dump(results, f, indent=2)

    results_path = os.path.join(RESULTS_DIR, f'xgboost_{subfolder}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

    print(f'Saved model artifacts to: {model_dir}')
    print(f'Saved summary results to: {results_path}')
    return results

In [ ]:
results_without_smote = train_xgboost_model(
    X_train,
    Y_train,
    X_test,
    Y_test,
    use_smote=False,
    n_trials=N_TRIALS,
)

results_without_smote

In [ ]:
results_with_smote = train_xgboost_model(
    X_train,
    Y_train,
    X_test,
    Y_test,
    use_smote=True,
    n_trials=N_TRIALS,
)

results_with_smote

In [ ]:
print('Finished. Results saved under:')
print(' -', RESULTS_DIR)
print(' -', MODEL_ROOT)